# 02 · 데이터 분할 검증 및 전처리

Stanford 40 Actions 중 8개 행동 데이터를 ResNet18 입력 형식으로 바꾸는 규칙을 정하고, 그 규칙이 제대로 작동하는지 확인합니다.
이 노트북에서 정한 전처리는 03_modeling의 세 학습 방식(Scratch / Feature Extraction / Fine-tuning)에 **똑같이** 적용합니다.

**확정한 설정**

| 항목 | 결정 | 이유 |
|---|---|---|
| 분할 | 제공된 Train 640 / Val 160 / Test 1,231 그대로 사용 | 튜터 제공 분할. 다시 나누지 않고 누수·라벨 연결만 검증 |
| 입력 크기 | 224×224 | ResNet18 기본 입력 |
| 크기 맞추기 | 비율 유지 Resize + Padding (ImageNet 평균색) | 종횡비가 넓게 분포 → 찌그러짐(Stretch)·가장자리 잘림(CenterCrop) 방지 |
| 정규화 | ImageNet 평균·표준편차 | 사전학습 가중치(IMAGENET1K_V1)의 입력 기준과 맞춤 |
| Train 증강 | 없음 (baseline) | 세 방식 비교를 깔끔하게 하고, 증강은 오분류 분석 기반 개선 실험 후보로 보관 |
| Val/Test | 무작위 요소 없는 고정 전처리 | 규칙 7 |
| Batch Size | 32 | 시작값 |

## 01 · 기본 설정

경로, 고정값(Seed), 전처리 설정값을 한곳에 모읍니다. 01_eda와 같은 방식으로 프로젝트 최상위 폴더를 자동으로 찾으므로 경로를 수정할 필요가 없습니다.

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import ResNet18_Weights

# ── 경로 설정: 01_eda와 같은 방식으로 프로젝트 최상위 폴더를 자동으로 찾음 ──
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw"   # metadata.csv, class_mapping.csv, images/ 가 들어 있는 폴더
FIG_DIR = PROJECT_ROOT / "figures"         # 보고서용 그림 저장 폴더 (Git에 올라감)
OUT_DIR = PROJECT_ROOT / "results"         # 설정 기록·분석용 표 저장 폴더 (Git에 올라감)
# 주의: data/ 폴더는 .gitignore로 제외되므로, 올려야 하는 결과물은 data/ 아래에 저장하지 않습니다.
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── 전처리 설정값 (03_modeling에서도 똑같이 사용) ──
SEED = 42          # 무작위 결과를 매번 똑같이 재현하기 위한 고정값
IMG_SIZE = 224     # ResNet18 기본 입력 크기
BATCH_SIZE = 32    # 한 번에 모델에 넣는 사진 수

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("프로젝트 위치:", PROJECT_ROOT)
print("데이터 폴더 존재 여부:", DATA_DIR.is_dir())
print("PyTorch", torch.__version__, "| GPU 사용 가능:", torch.cuda.is_available())

## 02 · 메타데이터 불러오기

CSV 파일 맨 앞에 BOM(눈에 보이지 않는 표시 문자)이 붙어 있어서, `encoding="utf-8-sig"`로 읽어야 첫 컬럼 이름이 `image_id`로 정상 인식됩니다.

In [ ]:
meta = pd.read_csv(DATA_DIR / "metadata.csv", encoding="utf-8-sig")
class_map = pd.read_csv(DATA_DIR / "class_mapping.csv", encoding="utf-8-sig").sort_values("target_id")

class_names = class_map["class_name"].tolist()        # 0~7번 순서대로 영어 클래스명
class_names_ko = class_map["class_name_ko"].tolist()  # 0~7번 순서대로 한국어 클래스명

print("메타데이터 크기:", meta.shape)
print("컬럼:", meta.columns.tolist())
display(class_map)

## 03 · 분할 검증 (다시 나누지 않고 확인만)

분할은 튜터가 제공한 그대로 사용합니다. 대신 아래 네 가지를 확인해 **데이터 누수**(평가용 사진이 학습에 섞이는 문제)와 **라벨 연결 오류**가 없는지 검증합니다.

1. 분할별·클래스별 장수가 예상(Train 80 / Val 20씩)과 같은가
2. 같은 사진이 두 분할에 동시에 들어 있지 않은가 (`image_id`, 원본 파일명 기준)
3. 라벨 번호(`target_id`)와 클래스 이름이 올바르게 짝지어져 있는가
4. 모든 이미지 파일이 실제로 있고, 문제없이 열리는가

In [ ]:
# 1) 분할별·클래스별 장수
count_table = pd.crosstab(meta["class_name_ko"], meta["split"])[["train", "val", "test"]]
display(count_table)

expected_total = {"train": 640, "val": 160, "test": 1231}
for split, n in expected_total.items():
    assert (meta["split"] == split).sum() == n, f"{split} 장수가 예상과 다릅니다"
assert (count_table["train"] == 80).all() and (count_table["val"] == 20).all(), "Train/Val 클래스별 장수가 예상과 다릅니다"
print("✔ 분할별·클래스별 장수 확인 완료")

# 2) 분할 사이 겹침 확인 (데이터 누수 점검)
splits = ["train", "val", "test"]
for col in ["image_id", "source_filename"]:
    groups = {s: set(meta.loc[meta["split"] == s, col]) for s in splits}
    for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
        overlap = groups[a] & groups[b]
        print(f"{col:16} | {a}-{b} 겹침: {len(overlap)}")
        assert len(overlap) == 0, f"{a}와 {b}에 같은 {col}가 있습니다"
print("✔ 분할 사이 겹침 없음")

# 3) 라벨 번호 ↔ 클래스 이름 연결 확인
id_to_name = dict(zip(class_map["target_id"], class_map["class_name"]))
assert (meta["target_id"].map(id_to_name) == meta["class_name"]).all(), "라벨 연결 오류가 있습니다"
print("✔ 라벨 번호와 클래스 이름 연결 정상")

In [ ]:
# 4) 파일 존재 + 정상적으로 열리는지 + 색상 모드 확인
problems = []          # 열리지 않거나 크기가 메타데이터와 다른 파일 목록
modes = []             # 각 사진의 색상 모드 (RGB, L(흑백), CMYK 등)

for row in meta.itertuples():
    path = DATA_DIR / row.image_path
    if not path.is_file():
        problems.append((row.image_id, "파일 없음"))
        modes.append(None)
        continue
    try:
        with Image.open(path) as img:
            img.load()                                   # 실제로 픽셀을 끝까지 읽어서 손상 여부 확인
            modes.append(img.mode)
            if img.size != (row.width, row.height):      # PIL의 size는 (가로, 세로)
                problems.append((row.image_id, f"크기 불일치 {img.size}"))
    except Exception as e:
        problems.append((row.image_id, f"열기 실패: {e}"))
        modes.append(None)

meta["mode"] = modes
print("문제 있는 파일 수:", len(problems))
for p in problems[:10]:
    print("  ", p)
print("색상 모드 분포:")
print(meta["mode"].value_counts(dropna=False))
# RGB가 아닌 사진(흑백 등)이 있어도 괜찮습니다. Dataset에서 convert("RGB")로 3채널로 통일합니다.

**확인할 것** · 겹침이 모두 0이고 문제 있는 파일이 0개면 통과입니다. RGB가 아닌 사진이 있다면 몇 장인지 기록해 두세요. 이 결과가 보고서의 "분할과 전처리 과정" 근거가 됩니다.

## 04 · 전처리 규칙 정의

`ResizePad`는 사진의 **긴 변을 224에 맞춰 비율을 유지한 채 줄이고**, 남는 공간을 ImageNet 평균색으로 채워 224×224 정사각형을 만듭니다.
평균색으로 채우는 이유: 정규화를 거치면 이 영역의 값이 거의 0이 되어, 모델에 "정보가 없는 영역"에 가깝게 전달됩니다.

Train 증강은 없음(baseline)으로 확정했으므로 `train_transform`은 `eval_transform`과 같습니다. 나중에 개선 실험에서 증강을 넣을 때는 `train_transform`만 바꾸면 됩니다.

In [ ]:
weights = ResNet18_Weights.IMAGENET1K_V1      # 사용할 사전학습 가중치 버전 (규칙 1)
MEAN = list(weights.transforms().mean)         # ImageNet 채널별 평균 [R, G, B]
STD = list(weights.transforms().std)           # ImageNet 채널별 표준편차 [R, G, B]
PAD_COLOR = tuple(round(m * 255) for m in MEAN)  # 평균값을 0~255 색으로 바꾼 것 → (124, 116, 104)


class ResizePad:
    """비율을 유지하며 긴 변을 size에 맞추고, 남는 곳을 fill 색으로 채워 정사각형으로 만든다."""

    def __init__(self, size=224, fill=PAD_COLOR):
        self.size = size
        self.fill = fill

    def __call__(self, img):
        w, h = img.size                                   # 원본 가로, 세로
        scale = self.size / max(w, h)                     # 긴 변이 size가 되도록 하는 배율
        new_w = max(1, round(w * scale))
        new_h = max(1, round(h * scale))
        img = img.resize((new_w, new_h), Image.Resampling.BILINEAR)   # 비율 유지 리사이즈
        canvas = Image.new("RGB", (self.size, self.size), self.fill)  # size x size 빈 도화지
        left = (self.size - new_w) // 2                   # 가운데에 놓기 위한 왼쪽 여백
        top = (self.size - new_h) // 2                    # 가운데에 놓기 위한 위쪽 여백
        canvas.paste(img, (left, top))
        return canvas

    def __repr__(self):
        return f"ResizePad(size={self.size}, fill={self.fill})"


# Validation / Test용 고정 전처리 (무작위 요소 없음, 규칙 7)
eval_transform = transforms.Compose([
    ResizePad(IMG_SIZE),
    transforms.ToTensor(),              # 0~255 픽셀 → 0~1 실수, 모양은 (채널, 세로, 가로)
    transforms.Normalize(MEAN, STD),    # (값 - 평균) / 표준편차 → ImageNet 기준에 맞춤
])

# Train용: baseline은 증강 없음 → eval과 동일
train_transform = eval_transform

print("ImageNet 평균:", MEAN, "| 표준편차:", STD, "| 패딩 색:", PAD_COLOR)
print(eval_transform)

## 05 · 세 가지 크기 맞추기 방식 비교 (시각화)

Resize+Padding을 고른 근거를 보여주기 위해, Train에서 **가장 길쭉한 사진 2장, 가장 홀쭉한 사진 2장, 무작위 2장**을 골라 세 방식을 나란히 비교합니다. 학습은 하지 않고 눈으로만 확인합니다. (01_eda의 Crop vs Pad 비교에 Stretch를 더해, 최종 전처리 함수로 다시 확인하는 단계입니다.)

- **Stretch**: 224×224로 그냥 늘리기 → 모양이 찌그러짐
- **CenterCrop**: 짧은 변을 224로 맞춘 뒤 가운데만 자르기 → 가장자리가 잘림
- **ResizePad**: 이번에 선택한 방식 → 모양·내용 보존, 대신 길쭉한 사진은 내용이 작아짐

In [ ]:
compare_methods = {
    "Stretch": transforms.Resize((IMG_SIZE, IMG_SIZE)),
    "CenterCrop": transforms.Compose([transforms.Resize(IMG_SIZE), transforms.CenterCrop(IMG_SIZE)]),
    "ResizePad": ResizePad(IMG_SIZE),
}

train_meta = meta[meta["split"] == "train"].copy()
train_meta["aspect_ratio"] = train_meta["width"] / train_meta["height"]   # 가로 ÷ 세로
picks = pd.concat([
    train_meta.nlargest(2, "aspect_ratio"),                 # 가장 가로로 긴 사진
    train_meta.nsmallest(2, "aspect_ratio"),                # 가장 세로로 긴 사진
    train_meta.sample(2, random_state=SEED),                # 무작위 사진
])

n_rows = 1 + len(compare_methods)   # 원본 1줄 + 방식별 1줄
fig, axes = plt.subplots(n_rows, len(picks), figsize=(2.6 * len(picks), 2.6 * n_rows))
for col, row in enumerate(picks.itertuples()):
    with Image.open(DATA_DIR / row.image_path) as img:
        img = img.convert("RGB")
        axes[0, col].imshow(img)
        axes[0, col].set_title(f"{row.class_name}\n{row.width}x{row.height} (ratio {row.aspect_ratio:.2f})", fontsize=8)
        for r, (name, method) in enumerate(compare_methods.items(), start=1):
            axes[r, col].imshow(method(img))
            axes[r, col].set_title(name, fontsize=8)
    for ax in axes[:, col]:
        ax.axis("off")
plt.tight_layout()
plt.savefig(FIG_DIR / "08_preprocessing_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

**확인할 것** · CenterCrop에서 활·노·원반이나 사람의 일부가 잘린 사진이 있나요? Stretch에서 사람이 눈에 띄게 찌그러졌나요? ResizePad에서 길쭉한 사진의 내용이 얼마나 작아졌나요? 관찰한 내용을 보고서의 전처리 선택 근거로 적어 두세요.

## 06 · Dataset과 DataLoader 만들기

- **Dataset**: 표의 i번째 행을 주면 (전처리된 사진, 정답 번호) 한 쌍을 돌려주는 규칙
- **DataLoader**: 이 규칙을 반복해 사진 32장씩 묶음(배치)으로 꺼내는 도구

모델 입력은 사진 픽셀뿐이고, 파일명·클래스명은 입력으로 쓰지 않습니다(규칙 4). Test DataLoader는 최종 모델이 정해진 뒤 03에서 한 번만 사용하므로 여기서는 만들지 않습니다(규칙 3).

In [ ]:
class ActionDataset(Dataset):
    def __init__(self, frame, transform):
        self.rows = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        with Image.open(DATA_DIR / row["image_path"]) as img:
            x = self.transform(img.convert("RGB"))   # 흑백·CMYK 사진도 RGB 3채널로 통일
        return x, int(row["target_id"])              # (사진 텐서, 정답 번호)


train_df = meta[meta["split"] == "train"].reset_index(drop=True)
val_df = meta[meta["split"] == "val"].reset_index(drop=True)

train_set = ActionDataset(train_df, train_transform)
val_set = ActionDataset(val_df, eval_transform)

g = torch.Generator().manual_seed(SEED)   # 배치를 섞는 순서도 재현 가능하게 고정
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, generator=g, num_workers=0)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
# num_workers=0: Windows 노트북 환경에서 오류 없이 돌아가도록 사진을 메인 프로세스에서 불러옴

images, labels = next(iter(train_loader))
print("Train", len(train_set), "장 | Val", len(val_set), "장")
print("배치 사진 모양:", tuple(images.shape), "| 정답 모양:", tuple(labels.shape), labels.dtype)
print("한 epoch 학습 배치 수:", len(train_loader), "| 검증 배치 수:", len(val_loader))
print(f"정규화 후 값 범위: {images.min():.2f} ~ {images.max():.2f}")
print("패딩 영역(왼쪽 위 모서리) 값:", images[0, :, 0, 0].round(decimals=3).tolist(), "← 패딩이 있는 사진이면 거의 0")

**확인할 것** · 배치 모양이 `(32, 3, 224, 224)`, 정답 모양이 `(32,)`이면 정상입니다. 640 ÷ 32 = 20이므로 학습 배치는 정확히 20개, 검증은 160 ÷ 32 = 5개입니다.

## 07 · 최종 입력을 사람 눈으로 다시 확인

모델에 실제로 들어가는 텐서를 정규화 이전 색으로 되돌려 보여줍니다. **사진과 정답 라벨이 맞게 짝지어졌는지** 마지막으로 확인하는 단계입니다.

In [ ]:
def denormalize(x):
    """정규화된 텐서를 화면에 보여줄 수 있는 0~1 색으로 되돌린다."""
    mean = torch.tensor(MEAN).view(3, 1, 1)
    std = torch.tensor(STD).view(3, 1, 1)
    return (x * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()   # (채널, 세로, 가로) → (세로, 가로, 채널)

fig, axes = plt.subplots(2, 4, figsize=(10, 5.5))
for ax, x, y in zip(axes.flat, images[:8], labels[:8]):
    ax.imshow(denormalize(x))
    ax.set_title(f"{int(y)} · {class_names[int(y)]}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.savefig(FIG_DIR / "09_preprocessed_batch.png", dpi=120, bbox_inches="tight")
plt.show()

## 08 · 전처리 설정과 사진별 정보 저장

- `preprocessing_config.json`: 03_modeling에서 같은 설정을 쓰고, 보고서에 학습 조건을 적기 위한 기록(규칙 2)
- `image_info.csv`: 사진별 종횡비와 **패딩 후 실제 내용이 차지하는 비율**. 나중에 오분류 분석에서 "길쭉한 사진(내용이 작아진 사진)에서 더 많이 틀리는가?"를 확인할 때 씁니다.

In [ ]:
# 사진별 정보: 종횡비와, ResizePad 후 224x224 중 실제 사진이 차지하는 면적 비율
info = meta[["image_id", "split", "target_id", "class_name", "width", "height"]].copy()
info["aspect_ratio"] = (info["width"] / info["height"]).round(3)
info["content_ratio"] = (info[["width", "height"]].min(axis=1) / info[["width", "height"]].max(axis=1)).round(3)
info.to_csv(OUT_DIR / "image_info.csv", index=False, encoding="utf-8-sig")

print("패딩 후 내용 비율 요약 (Train+Val):")
display(info[info["split"] != "test"]["content_ratio"].describe().round(3))

config = {
    "model_input_size": IMG_SIZE,
    "resize_method": "ResizePad: 비율 유지 Resize(긴 변=224) + 남는 영역 ImageNet 평균색 패딩",
    "pad_color_rgb": list(PAD_COLOR),
    "normalize_mean": MEAN,
    "normalize_std": STD,
    "pretrained_weights": "ResNet18_Weights.IMAGENET1K_V1",
    "train_augmentation": "없음 (baseline)",
    "eval_transform": "ResizePad → ToTensor → Normalize (Val/Test 고정)",
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "split_counts": meta["split"].value_counts().to_dict(),
    "non_rgb_images": int((meta["mode"] != "RGB").sum()),
    "problem_files": len(problems),
}
with open(OUT_DIR / "preprocessing_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print(json.dumps(config, ensure_ascii=False, indent=2))

## 정리

- 분할: 제공된 Train 640 / Val 160 / Test 1,231을 그대로 사용, 분할 간 겹침·라벨 연결·파일 손상 검증 완료
- 전처리: 224×224 비율 유지 Resize + ImageNet 평균색 Padding → ToTensor → ImageNet 정규화
- Train 증강: 없음(baseline). Val/Test: 고정 전처리
- 03_modeling에서는 04번(전처리 규칙)과 06번(Dataset) 코드를 그대로 가져가 세 학습 방식에 똑같이 적용합니다.